# 4D-Humans VR: Google Colab 3D Generation Server Runner (TRELLIS.2)

This notebook sets up the environment and launches the Microsoft **TRELLIS.2 Image-to-3D** generation server (4B parameters), tunneling it to your local frontend application using a secure Ngrok bridge.

### Step 1: Install Dependencies
Colab comes with PyTorch pre-installed. Run the cell below to install our server dependencies and the custom CUDA extensions required by the TRELLIS.2 pipeline.

In [ ]:
# Always reset the working directory dynamically depending on environment (Colab vs Kaggle)
import os
import shutil

base_dir = "/kaggle/working" if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ else "/content"
os.makedirs(base_dir, exist_ok=True)
os.chdir(base_dir)
print(f"Working directory reset to: {os.getcwd()}")

# 1. Clean up and clone your repository
if os.path.exists(os.path.join(base_dir, "BISAG")):
    shutil.rmtree(os.path.join(base_dir, "BISAG"))

!git clone https://github.com/anantj09/BISAG.git
os.chdir(os.path.join(base_dir, "BISAG", "Backend"))
print(f"Current backend directory: {os.getcwd()}")

# 2. Install base PyPI dependencies
!pip install -r colab/requirements_colab.txt

# 3. Install spconv and clone/setup Microsoft TRELLIS.2
!pip install spconv

# Clone TRELLIS.2 repository
if os.path.exists(os.path.join(base_dir, "TRELLIS")):
    shutil.rmtree(os.path.join(base_dir, "TRELLIS"))

!git clone --recursive https://github.com/microsoft/TRELLIS.2.git $base_dir/TRELLIS

# Install C++ and CUDA libraries (Eigen3 headers are required by o-voxel)
!sudo apt-get -y install libeigen3-dev

# Install CuMesh and FlexGEMM with no-build-isolation first (required by o-voxel)
!pip install git+https://github.com/JeffreyXiang/CuMesh.git --no-build-isolation
!pip install git+https://github.com/JeffreyXiang/FlexGEMM.git --no-build-isolation

# If the submodule clone failed or is incomplete, use system Eigen headers via symlink
eigen_submodule_path = os.path.join(base_dir, "TRELLIS", "o-voxel", "third_party", "eigen")
eigen_header_path = os.path.join(eigen_submodule_path, "Eigen")
if not os.path.exists(eigen_header_path):
    print("Submodule Eigen headers not found. Symlinking system Eigen...")
    if os.path.exists(eigen_submodule_path):
        if os.path.islink(eigen_submodule_path) or os.path.isdir(eigen_submodule_path):
            try:
                os.unlink(eigen_submodule_path)
            except OSError:
                shutil.rmtree(eigen_submodule_path)
    os.symlink("/usr/include/eigen3", eigen_submodule_path)

# Install o-voxel with no-build-isolation
ovoxel_path = os.path.join(base_dir, "TRELLIS", "o-voxel")
!pip install $ovoxel_path --no-build-isolation

# Install nvdiffrast using no-build-isolation
!pip install git+https://github.com/NVlabs/nvdiffrast.git --no-build-isolation

# Install diff-gaussian-rasterization for 3D Gaussian Splatting capabilities with no-build-isolation
!pip install git+https://github.com/graphdeco-inria/diff-gaussian-rasterization.git --no-build-isolation

### Step 2: Configure Hugging Face Token & Request Model Access
TRELLIS.2 uses Meta's **DINOv3** model (`facebook/dinov3-vitl16-pretrain-lvd1689m`) as a feature extractor, which is a gated repository.

1. Log in to Hugging Face, visit [facebook/dinov3-vitl16-pretrain-lvd1689m](https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m), and click **"Request Access"** (access is granted instantly).
2. Generate a **Read** token in your Hugging Face [Access Tokens settings](https://huggingface.co/settings/tokens).
3. Set your Token in your environment Secrets:
   * **On Google Colab:** Click the **Secrets (key icon 🔑)** in the left sidebar, add a secret named `HF_TOKEN` with your token value, and enable **Notebook access**.
   * **On Kaggle:** Go to **Add-ons > Secrets** in the top menu of your notebook, add a secret named `HF_TOKEN` with your token value, and click **Save**.

### Step 3: Open Tunnel and Start Server
1. Go to your [Ngrok Dashboard](https://dashboard.ngrok.com/) and copy your free authtoken.
2. Paste it in place of `YOUR_NGROK_AUTHTOKEN` in the code cell below.
3. Run the cell. Once the setup completes, copy the generated public HTTPS URL (e.g. `https://xxxx.ngrok-free.app`).
4. Open the Settings panel on your local page and paste this URL to activate Colab/Kaggle generation!

In [ ]:
import os

# 1. Try to load HF_TOKEN from Colab Secrets
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print("Successfully loaded HF_TOKEN from Google Colab Secrets!")
except Exception:
    pass

# 2. Try to load HF_TOKEN from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ['HF_TOKEN'] = token
        print("Successfully loaded HF_TOKEN from Kaggle User Secrets!")
except Exception:
    pass

if not os.environ.get('HF_TOKEN'):
    print("Warning: HF_TOKEN not found in Secrets. Gated model downloads may fail.")

# Launch the modular server and Ngrok tunnel
# We use python -u to disable stdout/stderr buffering, allowing real-time progress bars to be visible
ngrok_token = "YOUR_NGROK_AUTHTOKEN"
os.system(f"python -u -m colab.main_colab --ngrok-token {ngrok_token}")